<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/exams/final/FIN1_correlated_risk_simulation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Managing Research Projects

## Project 1: A Monte Carlo risk engine with correlated durations and risk events

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

The schedule network of Weeks 9 and 10 is extended with correlated task durations through a Gaussian copula, discrete risk events with probabilities and impacts, and task costs. The example reports percentiles of duration and cost, the criticality index of every task and the correlation of each task with the total [1, 2].

**Required extensions.** Add resource constraints or shared staff, risk responses whose cost can be compared with the risk they remove, and a calibration step that sets the distributions from historical data or expert ranges. Compare the resulting contingency with the deterministic plan and with PERT, and discuss the criticism of qualitative risk scoring [3, 4].

## What this example implements

The notebook simulates 20,000 projects for three scenarios, computes duration and cost percentiles, the criticality index of each task and a sensitivity ranking, and plots the distribution against the deterministic plan.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Correlated durations and risk events

In [ ]:
import pandas as pd
from scipy.stats import norm
rng = np.random.default_rng(SEED)
NET = [("A", "Literature review", 3, [], 20), ("B", "Data access", 2, [], 15), ("C", "Data collection", 5, ["B"], 40),
       ("D", "Method design", 4, ["A"], 30), ("E", "Prototype", 6, ["C", "D"], 60), ("F", "Benchmark design", 2, ["D"], 10),
       ("G", "Experiments", 4, ["E", "F"], 35), ("H", "Paper writing", 3, ["G"], 15), ("I", "Workshop", 2, ["G"], 12)]
RISKS = [("ethics approval delayed", 0.3, {"B": 3}), ("key student leaves", 0.15, {"E": 4, "G": 2}), ("hypothesis fails the first test", 0.25, {"G": 3})]

def simulate(n=20000, rho=0.0, low=0.9, high=1.6, risks=True):
    k = len(NET)
    cov = np.full((k, k), rho) + (1 - rho) * np.eye(k)
    u = norm.cdf(rng.multivariate_normal(np.zeros(k), cov, n))          # Gaussian copula for correlated durations
    dur = {}
    for j, (t, _, m, _, _) in enumerate(NET):
        a, b = low * m, high * m
        f = (m - a) / (b - a)
        dur[t] = np.where(u[:, j] < f, a + np.sqrt(u[:, j] * (b - a) * (m - a)), b - np.sqrt((1 - u[:, j]) * (b - a) * (b - m)))
    if risks:
        for _, p, impact in RISKS:
            hit = rng.random(n) < p
            for t, extra in impact.items():
                dur[t] = dur[t] + hit * extra
    ef = {}
    for t, _, _, pred, _ in NET:
        ef[t] = (np.max([ef[q] for q in pred], axis=0) if pred else np.zeros(n)) + dur[t]
    total = np.max(np.vstack(list(ef.values())), axis=0)
    cost = sum(dur[t] * c for t, _, _, _, c in NET)
    return total, cost, dur, ef

def criticality(dur, ef, total):
    lf = {}
    for t, _, _, _, _ in reversed(NET):
        succ = [s for s, _, _, p, _ in NET if t in p]
        lf[t] = np.min([lf[s] - dur[s] for s in succ], axis=0) if succ else total
    return {t: float(np.mean(lf[t] - ef[t] < 1e-9)) for t, *_ in NET}

rows = []
for label, rho, risks in [("independent, no risk events", 0.0, False), ("independent, risk events", 0.0, True), ("correlated 0.5, risk events", 0.5, True)]:
    total, cost, dur, ef = simulate(rho=rho, risks=risks)
    rows.append(dict(scenario=label, **{f"duration P{q}": np.percentile(total, q) for q in (50, 80, 95)}, **{f"cost P{q}": np.percentile(cost, q) for q in (50, 80)}))
pd.DataFrame(rows).set_index("scenario").round(1)

## 2. Criticality and sensitivity

In [ ]:
total, cost, dur, ef = simulate(rho=0.5)
ci = criticality(dur, ef, total)
sens = {t: float(np.corrcoef(dur[t], total)[0, 1]) for t in dur}
summary = pd.DataFrame({"criticality index": ci, "correlation with duration": sens}).round(3)
print(summary)
plt.hist(total, bins=60)
for q, c in [(50, "k"), (80, "r")]:
    plt.axvline(np.percentile(total, q), color=c)
plt.axvline(20, color="g", ls="--")
plt.xlabel("project duration (weeks)"); plt.title("Deterministic plan (dashed), P50 and P80"); plt.show()

## Report and submission

Both tracks follow the course report template. The report states a question, justifies the approach, presents evidence from the literature or from the simulation with figures, discusses limitations and ends with recommendations for managing research projects. Sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Vose, D. (2008). *Risk Analysis: A Quantitative Guide* (3rd ed.). Wiley.

[2] ISO (2018). *ISO 31000:2018 Risk management: Guidelines*. International Organization for Standardization.

[3] Hubbard, D. W. (2009). *The Failure of Risk Management: Why It's Broken and How to Fix It*. Wiley.

[4] Malcolm, D. G., Roseboom, J. H., Clark, C. E., & Fazar, W. (1959). Application of a technique for research and development program evaluation. *Operations Research*, 7(5), 646-669. <https://doi.org/10.1287/opre.7.5.646>